# SFJ Media Skills – EVT280: einmaliger Folgelauf zu EVT277/279

**Bedienung:** In Google Colab nur **Laufzeit → Alle ausführen** anklicken. Wenn Google die Anmeldung verlangt, mit dem Konto des Drive-Eigentümers bestätigen. Keine Eingaben oder API-Keys nötig.

Verarbeitet die **409 bereits protokollierten Quellenfehler** und **11’841 Belegkandidaten** aus dem erfolgreich abgeschlossenen kostenlosen EVT277-Gesamtlauf, ohne den 2’579er Quellenlauf zu wiederholen.

- **233** technisch für einen normalen, respektvollen Neuabruf geeignete URLs: kostenlos, mit robots.txt, HTTPS/DNS, Host-Abständen, Downloadlimiten und Checkpoints mit Drive-Readback.
- **176** andere URL-Holds: bereits geprüft/gesperrt/manuell zu entscheiden; nicht umgehen. Schliessungsstatus bleibt offen.
- **4’353** Angebots-Feldgruppen, **3’167** Belegkandidaten aus Sammelseiten als Quarantäne, alle anderen Feldkandidaten ohne automatische Promotion.
- Die **307 geschützten Archiv-/QA09-Angebote** werden nicht neu geöffnet.
- Keine automatischen Master-, Website-, Occurrence-, Main- oder Release-Writes; keine Firecrawl-Kosten.

Bei Unterbrechung einfach dasselbe Notebook erneut mit **Alle ausführen** starten: Bereits hash-/readback-bestätigte Zwischenstände werden übernommen. Nach Abschluss gibt Colab den direkten Google-Drive-Link zum Ergebnis-ZIP aus. **Die technische Nachprüfung kann fachlich offene Belege nicht automatisch in Website-Gold verwandeln.**


In [ ]:
%pip -q install requests beautifulsoup4 pymupdf google-api-python-client google-auth-httplib2


In [ ]:
#@title 1. Deterministische Prüf- und Quarantänefunktionen
"""SFJ EVT280 bulk follow-up: immutable, read-only offer/field/source triage.
No external I/O at import. Compatible with Colab Python 3.11 and local Python.
"""
from __future__ import annotations
import csv
import hashlib
import io
import json
import re
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlsplit

SOURCE_SHA='d1a4c36ba4a0771f114da18720a5e503702f349b73fb52139e8076a3adefd080'
SOURCE_FILE_ID='14eVKfokT7AzDGQOQMKRPGA5lK4e_roRz'
BIBLE_ID='1b1Zx5sjZuqYrEQr6zy9xlxs0eW7R4iDOpEBFl8ZeXsw'
EVT278_DOC='1ND75bkQRtWXbky_vCtCRQQ0-yoxCC_uVgR2abdX6Xzs'
EVT279_DOC='1JbDyZWn9fEdXlIOiL_KRfoXoAhQd6H38ooxdTDZb3UA'
EXPECTED_COUNTRIES={'CH':1223,'AT':124,'LI':5,'DE':111,'USA':2355}
PROTECTED_COUNT=307
PRE_REVIEWED_IDS={
  'CH:V78-01','CH:V78-02','CH:V78-03','CH:V78-05',
  'AT:AT-PROD-OFFER-173','CH:CH-V19-N02','AT:AT-PROD-OFFER-093',
  'CH:OFFER-FF-002','CH:OFFER-FF-005','CH:OFFER-FF-014',
  'CH:OFFER-FF-016','CH:OFFER-FF-018','CH:OFFER-BICT-001',
  'CH:OFFER-ATED-004','CH:OFFER-FHNW-011','CH:OFFER-FHNW-012',
}
# Recheck only where a normal public request could yield a different status.
# Never retry a robots, 403, security, rate-limit or oversized refusal.
ELIGIBLE_RECHECK={'HTTP_404','EMPTY_TEXT','CONTENT_INSUFFICIENT','HTTP_202','REDIRECT_INVALID'}
FORBIDDEN_RECHECK={'ROBOTS_NOT_ALLOWED_OR_UNKNOWN','REDIRECT_ROBOTS_NOT_ALLOWED',
                   'HTTP_403','DESTINATION_NOT_PUBLIC_OR_UNRESOLVED','HTTP_429',
                   'CONTENT_TOO_LARGE'}
EXPECTED_FIELDS={'DELIVERY','DURATION','PRICE','ACCESS','DATES'}


def sha(data):
    if isinstance(data,str):data=data.encode('utf-8')
    return hashlib.sha256(data).hexdigest()


def jsonl(objs):
    return ''.join(json.dumps(r,ensure_ascii=False,sort_keys=True,separators=(',',':'))+'\n' for r in objs).encode('utf-8')


def csv_bytes(rows,fields):
    b=io.StringIO(newline='')
    w=csv.DictWriter(b,fieldnames=list(fields),delimiter='\t',extrasaction='ignore',lineterminator='\n')
    w.writeheader()
    for row in rows:
        safe={}
        for key in fields:
            value=row.get(key,'')
            if isinstance(value,(list,dict)):value=json.dumps(value,ensure_ascii=False)
            if value is None:value=''
            value=str(value)
            # Files can be imported into Google Sheets; avoid formula execution.
            if value.lstrip().startswith(('=','+','-','@')):value="'"+value
            safe[key]=value
        w.writerow(safe)
    return b.getvalue().encode('utf-8')


def zip_bytes(entries):
    output=io.BytesIO()
    with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for name,content in sorted(entries.items()):
            if name.startswith('/') or '..' in name.split('/') or not name or name in ('','.'):raise ValueError('Unsafe ZIP member')
            if isinstance(content,str):content=content.encode('utf-8')
            zi=zipfile.ZipInfo(name,date_time=(2026,11,7,0,0,0))
            zi.compress_type=zipfile.ZIP_DEFLATED
            z.writestr(zi,content)
    return output.getvalue()


def unpack_zip(data):
    if not zipfile.is_zipfile(io.BytesIO(data)):raise ValueError('Not a ZIP')
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        if z.testzip() is not None:raise ValueError('ZIP CRC error')
        names=z.namelist()
        if len(names)!=len(set(names)):raise ValueError('Duplicate ZIP member')
        if any(name.startswith('/') or '..' in name.split('/') for name in names):raise ValueError('Unexpected archive path')
        return {name:z.read(name) for name in names}


def read_csv(data,delimiter='\t'):
    return list(csv.DictReader(io.StringIO(data.decode('utf-8-sig')),delimiter=delimiter))


def read_jsonl(data):
    return [json.loads(line) for line in data.decode('utf-8').splitlines() if line.strip()]


def parse_id_array(value):
    try:ids=json.loads(value)
    except Exception as e:raise ValueError('IDs not valid JSON') from e
    if not isinstance(ids,list) or not ids or not all(isinstance(i,str) and re.fullmatch(r'(CH|AT|LI|DE|USA):[^:]+',i) for i in ids):
        raise ValueError('Malformed country+offer IDs')
    if len(ids)!=len(set(ids)):raise ValueError('Duplicate IDs')
    return ids


def load_verified_source(payload):
    if sha(payload)!=SOURCE_SHA:raise ValueError('EVT277 source ZIP SHA256 mismatch; abort')
    files=unpack_zip(payload)
    required={'konsolidierter_3818_grunstock.jsonl','quellenstatus.tsv',
              'belegkandidaten_mit_originaltext.tsv','restfaelle_und_holds.tsv','kennzahlen.json'}
    if not required<=set(files):raise ValueError('Missing required source members: '+str(required-set(files)))
    base=read_jsonl(files['konsolidierter_3818_grunstock.jsonl'])
    status=read_csv(files['quellenstatus.tsv'])
    candidates=read_csv(files['belegkandidaten_mit_originaltext.tsv'])
    unresolved=read_csv(files['restfaelle_und_holds.tsv'])
    meta=json.loads(files['kennzahlen.json'])
    ids={f"{r['market']}:{r['id']}" for r in base}
    if len(base)!=3818 or len(ids)!=3818:raise ValueError('3818 unique base IDs not found')
    countries=Counter(r['market'] for r in base)
    if dict(countries)!=EXPECTED_COUNTRIES:raise ValueError('Country-count drift: '+str(countries))
    protected={f"{r['market']}:{r['id']}" for r in base if r.get('protectedNoReopen')}
    # Frozen source contract: baseline has 145 flags; full archive/QA09 are tracked separately.
    if len(protected)!=145 or meta.get('no_reopen_unique_ids')!=PROTECTED_COUNT:
        raise ValueError('No-reopen policy count drift')
    if len(status)!=2579 or len(unresolved)!=409 or len(candidates)!=11841:
        raise ValueError('EVT277 source/candidate/hold count drift')
    statuses={r['url']:r for r in status}
    if len(statuses)!=len(status):raise ValueError('Duplicate source URLs')
    unresolved_urls={r['url'] for r in unresolved}
    if len(unresolved_urls)!=409:raise ValueError('Duplicate unresolved source URLs')
    if any(url not in statuses for url in unresolved_urls):raise ValueError('Unknown unresolved source')
    for r in status:
        r['id_list']=parse_id_array(r['ids'])
        if not set(r['id_list'])<=ids:raise ValueError('Foreign offer ID in source statuses')
        if set(r['id_list']) & protected:raise ValueError('Protected offer included in source work')
        if r['url'] in unresolved_urls and r['status']=='CONTENT_OK':raise ValueError('Unresolved URL marked CONTENT_OK')
    for r in unresolved:
        r['id_list']=parse_id_array(r['ids'])
        if sorted(r['id_list'])!=sorted(statuses[r['url']]['id_list']) or r['reason']!=statuses[r['url']]['status']:
            raise ValueError('Unresolved-source mismatch: '+r['url'])
    for r in candidates:
        url=r['source_url']
        if url not in statuses or statuses[url]['status']!='CONTENT_OK':raise ValueError('Unconfirmed source candidate')
        if r['country_id'] not in statuses[url]['id_list']:raise ValueError('Candidate has wrong offer assignment')
        if r['source_body_sha256']!=statuses[url]['sha256'] or not r['source_body_sha256']:
            raise ValueError('Candidate source SHA mismatch')
        if r['country_id'] in protected:raise ValueError('Protected candidate')
        if r['field'] not in EXPECTED_FIELDS:raise ValueError('Unknown candidate field')
    if meta.get('paid_results')!=0 or meta.get('published') is not False:
        raise ValueError('Source was not a free unpublished run')
    if not PRE_REVIEWED_IDS <= ids:raise ValueError('EVT279 source IDs missing')
    return {'base':base,'statuses':status,'status_by_url':statuses,
            'unresolved':unresolved,'candidates':candidates,'metadata':meta,
            'protected':protected,'offer_ids':ids}


def route_candidates(src):
    candidates=src['candidates'];statuses=src['status_by_url']
    shared_count=Counter(); group_to_ids=defaultdict(set); groups=defaultdict(list)
    for row in candidates:
        key=(row['source_url'],row['field'],row['excerpt'].strip())
        group_to_ids[key].add(row['country_id'])
    detail=[]
    for r in candidates:
        st=statuses[r['source_url']]
        cross=len(st['id_list'])>1
        collision=len(group_to_ids[(r['source_url'],r['field'],r['excerpt'].strip())])>1
        route='SHARED_SOURCE_QUARANTINE' if cross else 'SINGLE_OFFER_SOURCE_REVIEW'
        d={**r,'source_offer_count':len(st['id_list']),
           'same_excerpt_cross_offer':'YES' if collision else 'NO',
           'route':route,'reviewed_evt279':'YES' if r['country_id'] in PRE_REVIEWED_IDS else 'NO',
           'field_write_allowed':'NO','occurrence_write_allowed':'NO','release':'BLOCKED'}
        detail.append(d)
        groups[(r['country_id'],r['field'])].append(d)
    field_routes=[]
    for (key,field),rr in sorted(groups.items()):
        excerpts={r['excerpt'].strip() for r in rr}
        urls={r['source_url'] for r in rr}
        quarantined=sum(r['route']=='SHARED_SOURCE_QUARANTINE' for r in rr)
        route='SHARED_SOURCE_IDENTITY_REQUIRED' if quarantined else 'EXACT_OFFER_EDITION_FIELD_REVIEW_REQUIRED'
        field_routes.append({'country_id':key,'field':field,'candidate_rows':len(rr),
                             'distinct_source_urls':len(urls),'distinct_excerpts':len(excerpts),
                             'shared_source_candidate_rows':quarantined,
                             'evt279_reviewed':'YES' if key in PRE_REVIEWED_IDS else 'NO',
                             'route':route,'field_write_allowed':'NO','release':'BLOCKED'})
    if len(detail)!=11841:raise ValueError('Candidate triage loss')
    # This was the exact shared-source count in EVT278, never infer it from excerpt collisions alone.
    if sum(r['route']=='SHARED_SOURCE_QUARANTINE' for r in detail)!=3167:
        raise ValueError('EVT278 3167 shared source guard drift')
    return detail,field_routes


def route_unresolved(src):
    output=[]
    for r in src['unresolved']:
        market={i.split(':',1)[0] for i in r['id_list']}
        if len(market)!=1:raise ValueError('Mixed market source, manual review required')
        m=next(iter(market))
        is_p0=any(i in {'CH:V78-01','CH:V78-02','CH:V78-03','CH:V78-05','AT:AT-PROD-OFFER-173'} for i in r['id_list'])
        priority='P0_GATE' if is_p0 else ('P1_CH_AT_LI' if m in {'CH','AT','LI'} else 'P2_DE_USA')
        reviewed=bool(set(r['id_list']) & PRE_REVIEWED_IDS)
        allowed=r['reason'] in ELIGIBLE_RECHECK and not reviewed
        route=('EVT279_ALREADY_TRIAGED_MANUAL_HOLD' if reviewed else
               'ORDINARY_PUBLIC_RECHECK_ALLOWED' if allowed else 'NO_RECHECK_MANUAL_HOLD')
        output.append({'url':r['url'],'country_ids':r['id_list'],'source_status_before':r['reason'],
                       'country':m,'priority':priority,'evt279_already_reviewed':reviewed,
                       'normal_recheck_allowed':allowed,'route':route,
                       'source_status_after':'NOT_RECHECKED','source_http_code':'',
                       'source_final_url':'','new_body_sha256':'','page_title':'',
                       'field_write_allowed':'NO','release':'BLOCKED'})
    # For automation, all failures are triaged, not falsely "solved".
    if len(output)!=409 or len({r['url'] for r in output})!=409:
        raise ValueError('Unresolved routing lost a URL')
    for r in output:
        if r['source_status_before'] in FORBIDDEN_RECHECK and r['normal_recheck_allowed']:
            raise ValueError('Blocked URL scheduled for automated recheck')
    return output


def assemble_report(src,source_routes,field_routes,candidate_routes,results):
    next_status=[]
    for row in source_routes:
        r=dict(row)
        now=results.get(r['url'])
        if now:
            if not r['normal_recheck_allowed']:raise ValueError('Forbidden source rechecked')
            if now.get('url')!=r['url']:raise ValueError('Result mismatched source URL')
            r.update(source_status_after=now['status'],source_http_code=now.get('http_code',''),
                     source_final_url=now.get('final_url',''),new_body_sha256=now.get('body_sha256',''),
                     page_title=now.get('page_title',''))
        next_status.append(r)
    if len(next_status)!=409:raise ValueError('Source result count drift')
    counts=Counter(r['source_status_after'] for r in next_status)
    report={'phase':'EVT280_BULK_TRIAGE_NO_AUTOMATIC_PROMOTION','source_sha256':SOURCE_SHA,
            'source_record_count':3818,'problem_urls':409,'candidate_rows':11841,
            'field_review_groups':len(field_routes),'source_recheck_eligible':sum(r['normal_recheck_allowed'] for r in next_status),
            'source_rechecks_done':len(results),'new_status_counts':dict(counts),
            'possibly_restored_source_urls':counts.get('CONTENT_OK',0),
            'shared_source_quarantined_candidate_rows':sum(r['route']=='SHARED_SOURCE_QUARANTINE' for r in candidate_routes),
            'single_offer_candidate_rows':sum(r['route']=='SINGLE_OFFER_SOURCE_REVIEW' for r in candidate_routes),
            'protected_archived_qa09_unique':PROTECTED_COUNT,
            'evt279_reviewed_offer_ids':len(PRE_REVIEWED_IDS),
            'auto_field_writes':0,'auto_occurrence_writes':0,'master_or_website_writes':0,'published':False,
            'needs_human_decision':True,'free_no_api_key':True,'automatic_gold_claim':False,
            'as_of':max((r.get('fetched_at','') for r in results.values()),default=src['metadata'].get('completed_at',''))}
    if report['shared_source_quarantined_candidate_rows']!=3167:raise ValueError('Shared evidence count drift')
    if report['single_offer_candidate_rows']!=8674:raise ValueError('Unshared evidence count drift')
    return next_status,report


def make_output(src,source_routes,field_routes,candidate_routes,results):
    next_status,report=assemble_report(src,source_routes,field_routes,candidate_routes,results)
    source_cols=['url','country_ids','country','priority','source_status_before','evt279_already_reviewed',
                 'normal_recheck_allowed','route','source_status_after','source_http_code',
                 'source_final_url','new_body_sha256','page_title','field_write_allowed','release']
    detail_cols=['country_id','field','source_url','source_checked_at','source_body_sha256','excerpt',
                 'identity_signal','decision','source_offer_count','same_excerpt_cross_offer','route',
                 'reviewed_evt279','field_write_allowed','occurrence_write_allowed','release']
    field_cols=['country_id','field','candidate_rows','distinct_source_urls','distinct_excerpts',
                'shared_source_candidate_rows','evt279_reviewed','route','field_write_allowed','release']
    readme=("EVT280: Source remediation / field identity review, not website gold.\n"
            "409 source failures all routed; only eligible public original URLs retried, never robots/403/security/429.\n"
            "11,841 evidence excerpts triaged; 3,167 shared-source candidates quarantined.\n"
            "No automatic field, occurrence, programme, master, GitHub main, or publication changes.\n"
            "External website content may remain unavailable; hold until independently evidenced.\n"
            "EVT279 already-reviewed offer IDs remain review holds and are not retried.\n")
    package=zip_bytes({'MANIFEST_EVT280.json':json.dumps(report,ensure_ascii=False,indent=2,sort_keys=True),
                       'QUELLEN_409_TRIAGE.tsv':csv_bytes(next_status,source_cols),
                       'FELDGRUPPEN_REVIEW.tsv':csv_bytes(field_routes,field_cols),
                       'BELEGKANDIDATEN_11841_GATES.tsv':csv_bytes(candidate_routes,detail_cols),
                       'RECHECK_ERGEBNISSE.jsonl':jsonl(sorted(results.values(),key=lambda r:r['url'])),
                       'README_READ_ONLY.txt':readme})
    out=unpack_zip(package)
    assert len(read_csv(out['QUELLEN_409_TRIAGE.tsv']))==409
    assert len(read_csv(out['BELEGKANDIDATEN_11841_GATES.tsv']))==11841
    assert len(read_csv(out['FELDGRUPPEN_REVIEW.tsv']))==len(field_routes)
    return package,report


In [ ]:
#@title 2. Projektstand verifizieren und vollständige Prüf-Queue laden
import io, json, hashlib, time, re, threading, socket, ipaddress, zipfile
from collections import Counter
from urllib.parse import urlsplit, urljoin
from urllib.robotparser import RobotFileParser
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload
from IPython.display import display, Markdown

auth.authenticate_user()  # Google may ask for consent: only interaction needed beyond Run all.
credentials, _ = google.auth.default()  # Match the authenticated Google Drive context used by successful EVT277.
drive=build('drive','v3',credentials=credentials,cache_discovery=False)
docs=build('docs','v1',credentials=credentials,cache_discovery=False)


def download_file(file_id):
    stream=io.BytesIO()
    media=drive.files().get_media(fileId=file_id)
    dl=MediaIoBaseDownload(stream,media,chunksize=1024*1024)
    done=False
    while not done:
        _,done=dl.next_chunk(num_retries=3)
    return stream.getvalue()


def doc_text(obj):
    if isinstance(obj,dict):
        if isinstance(obj.get('textRun'),dict):
            yield obj['textRun'].get('content','')
        for k,v in obj.items():
            if k!='textRun':yield from doc_text(v)
    elif isinstance(obj,list):
        for e in obj:yield from doc_text(e)


def get_docs_text(file_id):
    return ''.join(doc_text(docs.documents().get(documentId=file_id,includeTabsContent=True).execute()))

bible=get_docs_text(BIBLE_ID)
events=[int(n) for n in re.findall(r'EVT-BIBEL-20261107-(\d+)',bible)]
if not events or max(events)<279:
    raise RuntimeError('Drive-Projektbibel enthält EVT279 noch nicht – keine automatische Arbeit')
previous=get_docs_text(EVT279_DOC)
if not all(k in previous for k in PRE_REVIEWED_IDS):
    raise RuntimeError('EVT279-Entscheide nicht vollständig im autoritativen Google Doc')
prior=get_docs_text(EVT278_DOC)
if not all(p in prior for p in ('409','11’841')):
    raise RuntimeError('EVT278-Quelle widersprüchlich: 409/11’841 fehlen')

source_payload=download_file(SOURCE_FILE_ID)
src=load_verified_source(source_payload)
candidate_routes,field_routes=route_candidates(src)
source_routes=route_unresolved(src)
eligible=sorted([r for r in source_routes if r['normal_recheck_allowed']],
                key=lambda r:({'P0_GATE':0,'P1_CH_AT_LI':1,'P2_DE_USA':2}[r['priority']],r['url']))
assert len(eligible)==233 and len(source_routes)==409 and len(candidate_routes)==11841
print('Autoritative Projektbibel:',max(events))
print('EVT277 Ergebnis-SHA:',SOURCE_SHA)
print('Quelle/Belege geprüft: 3’818 IDs / 409 Problem-URLs / 11’841 Kandidaten')
print('Feldprüfgruppen:',len(field_routes),'· unzulässige Mehrangebotsübertragung:',3167)
print('Automatisch ohne Sperrumgehung nachprüfbare URLs:',len(eligible))

# Existing successful EVT277 folder. Create only a child checkpoint folder; no master writes.
PARENT_FOLDER_ID='1-_y7j_hZfWg9vWLbWDl3wl7vOxRcktgh'
OUT_FOLDER_NAME='SFJ_EVT280_ONECLICK_REMEDIATION_'+SOURCE_SHA[:12]
def list_child_files(folder_id):
    token=None;files=[]
    while True:
        resp=drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken,files(id,name,size,mimeType)',pageSize=1000,pageToken=token).execute()
        files.extend(resp.get('files',[]));token=resp.get('nextPageToken')
        if not token:break
    return files

folders=[f for f in list_child_files(PARENT_FOLDER_ID)
         if f['name']==OUT_FOLDER_NAME and f.get('mimeType')=='application/vnd.google-apps.folder']
if len(folders)>1:raise RuntimeError('Doppelte EVT280-Laufordner; keine unsichere Wahl')
if folders:out_folder_id=folders[0]['id']
else:
    out_folder_id=drive.files().create(body={'name':OUT_FOLDER_NAME,'parents':[PARENT_FOLDER_ID],
        'mimeType':'application/vnd.google-apps.folder'},fields='id').execute()['id']
print('Checkpointordner:',f'https://drive.google.com/drive/folders/{out_folder_id}')


def write_once(name,data):
    """Immutable file. Persisted bytes are verified via a new Google Drive readback."""
    if not name.endswith('.zip'):raise ValueError('Only ZIP output')
    old=[f for f in list_child_files(out_folder_id) if f['name']==name]
    if len(old)>1:raise RuntimeError('Mehrfacher Drive-Name: '+name)
    if old:
        if sha(download_file(old[0]['id']))!=sha(data):raise RuntimeError('Drive-Live-Konflikt: '+name)
        return old[0]['id']
    media=MediaIoBaseUpload(io.BytesIO(data),mimetype='application/zip',resumable=False)
    file_id=drive.files().create(body={'name':name,'parents':[out_folder_id]},
        media_body=media,fields='id').execute()['id']
    if sha(download_file(file_id))!=sha(data):
        raise RuntimeError('Drive-Readback/SHA-Konflikt: '+name)
    return file_id



In [ ]:
#@title 3. Alle zulässigen Quellen nachprüfen – kostenlos und mit Wiederaufnahme
MAX_WORKERS=6
BATCH_SIZE=25
MAX_BYTES=6_000_000
MAX_TEXT=85_000
USER_AGENT='SFJ-MediaSkills-Research/1.1 (official data audit; no automated publication)'

# Avoid accidental unsafe hosts and redirect destinations. Never crawl robots/403 from original run.
def allowed_address(url):
    p=urlsplit(url)
    if p.scheme!='https' or not p.hostname or p.username or p.password:
        return False
    try:
        if p.port not in (None,443):return False
        addresses=socket.getaddrinfo(p.hostname,443,type=socket.SOCK_STREAM)
        return bool(addresses) and all(ipaddress.ip_address(a[4][0]).is_global for a in addresses)
    except (OSError,ValueError):return False

robot_lock=threading.Lock()
robot_host_locks={}
robot_cache={}
host_pace_lock=threading.Lock()
host_next={}
host_banned=set()


def host_delay(url):
    host=urlsplit(url).hostname
    with host_pace_lock:
        target=max(time.monotonic(),host_next.get(host,0))
        delay=target-time.monotonic()
        host_next[host]=target+2.0
    if delay>0:time.sleep(delay)


def raw_get(url,stream=False):
    host_delay(url)
    return requests.get(url,timeout=(8,20),headers={
        'User-Agent':USER_AGENT,'Accept':'text/html,text/plain,application/pdf'},
        allow_redirects=False,stream=stream)


def robots_allowed(url):
    p=urlsplit(url)
    origin=f'{p.scheme}://{p.netloc}'
    with robot_lock:lock=robot_host_locks.setdefault(origin,threading.Lock())
    with lock:
        if origin in robot_cache:
            rp=robot_cache[origin]
            return bool(rp and rp.can_fetch(USER_AGENT,url))
        rp=RobotFileParser();current=origin+'/robots.txt'
        try:
            for _ in range(4):
                if not allowed_address(current):
                    robot_cache[origin]=False;return False
                with raw_get(current) as resp:
                    code=resp.status_code
                    if code in (301,302,303,307,308):
                        next_url=urljoin(current,resp.headers.get('Location',''))
                        if not allowed_address(next_url):
                            robot_cache[origin]=False;return False
                        current=next_url;continue
                    if code==200:
                        if len(resp.content)>512000:
                            robot_cache[origin]=False;return False
                        rp.parse(resp.text.splitlines());robot_cache[origin]=rp
                        return rp.can_fetch(USER_AGENT,url)
                    if code in (404,410):
                        rp.parse([]);robot_cache[origin]=rp;return True
                    robot_cache[origin]=False;return False
        except (requests.RequestException,ValueError,OSError):
            robot_cache[origin]=False;return False
        robot_cache[origin]=False;return False


def check_url(row):
    url=row['url']
    if not row['normal_recheck_allowed']:
        raise RuntimeError('Attempted to bypass a protected original source status')
    out={'url':url,'status':'UNKNOWN','http_code':'','final_url':'',
         'body':'','body_sha256':'','page_title':'','canonical_candidate':'',
         'fetched_at':datetime.now(timezone.utc).isoformat(),'note':'','method':'EVT280_FREE_DIRECT'}
    try:
        if not allowed_address(url):out['status']='DESTINATION_NOT_PUBLIC_OR_UNRESOLVED';return out
        current=url
        for _ in range(5):
            host=urlsplit(current).hostname
            with host_pace_lock:
                if host in host_banned:
                    out['status']='HOST_RATE_LIMIT_OR_DENIAL';return out
            if not robots_allowed(current):
                out['status']='ROBOTS_NOT_ALLOWED_OR_UNKNOWN';return out
            with raw_get(current,stream=True) as resp:
                code=resp.status_code
                out['http_code']=code;out['final_url']=current
                if code in (301,302,303,307,308):
                    next_url=urljoin(current,resp.headers.get('Location',''))
                    if not allowed_address(next_url) or next_url==current:
                        out['status']='REDIRECT_INVALID';return out
                    current=next_url;continue
                if code in (403,429):
                    with host_pace_lock:host_banned.add(host)
                if code!=200:
                    out['status']='HTTP_'+str(code);return out
                ctype=resp.headers.get('Content-Type','').lower()
                allowed=('text/','application/pdf','application/xhtml+xml')
                if ctype and not ctype.startswith(allowed):
                    out['status']='UNSUPPORTED_CONTENT_TYPE';return out
                fragments=[];size=0
                for chunk in resp.iter_content(chunk_size=65536):
                    size+=len(chunk)
                    if size>MAX_BYTES:
                        out['status']='CONTENT_TOO_LARGE';return out
                    fragments.append(chunk)
                raw=b''.join(fragments)
                if ctype.startswith('application/pdf') or raw.startswith(b'%PDF'):
                    import fitz
                    with fitz.open(stream=raw,filetype='pdf') as pdf:
                        body='\n'.join(pdf[i].get_text() for i in range(min(len(pdf),50)))
                elif 'html' in ctype or b'<html' in raw[:3000].lower():
                    soup=BeautifulSoup(raw,'html.parser')
                    out['page_title']=soup.title.get_text(' ',strip=True)[:300] if soup.title else ''
                    for rel in soup.find_all('link',rel=True):
                        if 'canonical' in [v.lower() for v in rel.get('rel',[])]:
                            candidate=urljoin(current,rel.get('href',''))
                            if allowed_address(candidate) and urlsplit(candidate).hostname==urlsplit(current).hostname:
                                out['canonical_candidate']=candidate
                            break
                    for bad in soup(['script','style','noscript','nav','footer']):bad.decompose()
                    main=soup.select_one('main,article,[role="main"]') or soup.body or soup
                    body=main.get_text('\n',strip=True)
                else:body=raw.decode('utf-8',errors='replace')
                body=body[:MAX_TEXT]
                out['body']=body;out['body_sha256']=sha(body)
                out['status']='CONTENT_OK' if len(body)>=220 else ('EMPTY_TEXT' if not body.strip() else 'CONTENT_INSUFFICIENT')
                return out
        out['status']='TOO_MANY_REDIRECTS';return out
    except Exception as exc:
        out['status']='FETCH_FAILED';out['note']=type(exc).__name__+': '+str(exc)[:120]
        return out


def checkpoint_records(blob):
    z=unpack_zip(blob)
    expected={'RECHECK_BATCH_MANIFEST.json','RECHECK_RESULTS.jsonl'}
    if set(z)!=expected:raise ValueError('Incorrect checkpoint member set')
    manifest=json.loads(z['RECHECK_BATCH_MANIFEST.json'])
    if manifest['source_sha256']!=SOURCE_SHA:
        raise RuntimeError('Untrusted checkpoint parent SHA')
    rows=read_jsonl(z['RECHECK_RESULTS.jsonl'])
    if len(rows)!=len(manifest['urls']) or set(r['url'] for r in rows)!=set(manifest['urls']):
        raise ValueError('Checkpoint membership mismatch')
    if len({r['url'] for r in rows})!=len(rows):raise ValueError('Duplicate URL in checkpoint')
    for r in rows:
        if r['status']=='CONTENT_OK':
            if len(r.get('body',''))<220 or sha(r['body'])!=r.get('body_sha256'):
                raise ValueError('Invalid CONTENT_OK checkpoint body')
        elif r.get('body') and sha(r['body'])!=r.get('body_sha256'):
            raise ValueError('Unverifiable checkpoint body')
    return manifest,rows

eligible_by_url={r['url']:r for r in eligible}
completed={}
files=list_child_files(out_folder_id)
for item in sorted(files,key=lambda x:x['name']):
    if re.fullmatch(r'RECHECK_EVT280_[a-f0-9]{12}_[a-f0-9]{16}\.zip',item['name']):
        manifest,records=checkpoint_records(download_file(item['id']))
        expected_name='RECHECK_EVT280_'+SOURCE_SHA[:12]+'_'+sha('\n'.join(sorted(manifest['urls'])))[:16]+'.zip'
        if item['name']!=expected_name:raise ValueError('Checkpoint filename/source mismatch')
        for r in records:
            if r['url'] not in eligible_by_url:raise RuntimeError('Checkpoint has forbidden source URL')
            if r['url'] in completed and completed[r['url']]!=r:
                raise RuntimeError('Incompatible duplicate source result')
            completed[r['url']]=r

print('Bereits readback-bestätigt:',len(completed),'von',len(eligible))
remaining=[r for r in eligible if r['url'] not in completed]
for start in range(0,len(remaining),BATCH_SIZE):
    batch=remaining[start:start+BATCH_SIZE]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        values=list(executor.map(check_url,batch))
    batch_urls=sorted(r['url'] for r in batch)
    manifest={'version':'EVT280','source_sha256':SOURCE_SHA,'urls':batch_urls,
              'count':len(batch),'no_master_writes':True}
    filename='RECHECK_EVT280_'+SOURCE_SHA[:12]+'_'+sha('\n'.join(batch_urls))[:16]+'.zip'
    payload=zip_bytes({'RECHECK_BATCH_MANIFEST.json':json.dumps(manifest,ensure_ascii=False,sort_keys=True),
                       'RECHECK_RESULTS.jsonl':jsonl(values)})
    checkpoint_records(payload)  # validate before uploading
    write_once(filename,payload)   # Google Drive write + SHA readback
    completed.update({r['url']:r for r in values})
    print('Gesichert:',len(completed),'/',len(eligible),'geeignete Quellen')

if len(completed)!=len(eligible):raise RuntimeError('Incomplete batch; resume notebook')



In [ ]:
#@title 4. Alles konsolidieren, auf Drift prüfen und als ZIP in Drive sichern
package,summary=make_output(src,source_routes,field_routes,candidate_routes,completed)
# Internal consistency checks before the one immutable final write.
if summary['source_rechecks_done']!=233 or summary['field_review_groups']!=4353:
    raise RuntimeError('Unexpected count in finale submission')
if summary['auto_field_writes'] or summary['published']:
    raise RuntimeError('Publication/write protection violated')
archive=unpack_zip(package)
assert len(read_csv(archive['QUELLEN_409_TRIAGE.tsv']))==409
assert len(read_csv(archive['BELEGKANDIDATEN_11841_GATES.tsv']))==11841

final_name='EVT280_ABSCHLUSS_READ_ONLY_'+SOURCE_SHA[:12]+'.zip'
final_id=write_once(final_name,package)
print('\nTECHNISCHER EVT280-FOLGELAUF ABGESCHLOSSEN. KEINE MASTER- ODER WEBSITE-WRITES.')
print('Drive-Ergebnis:',f'https://drive.google.com/file/d/{final_id}/view')
print('Drive-Ordner:',f'https://drive.google.com/drive/folders/{out_folder_id}')
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(Markdown('**Wichtig:** Die Quelle-Erreichbarkeit wird geprüft; ohne angebots- und feldgenaue Primärbelege bleiben alle entsprechenden Felder auf HOLD. Keine automatische Publikation.'))


**Fertig nach Ausgabe des Google-Drive-Ergebnislinks.** Bereits bearbeitete URL-Checkpoints bleiben gespeichert. Noch offene fachliche Fälle bleiben auf HOLD und werden im Abschluss-ZIP einzeln ausgewiesen. Kein automatischer Website-Release.